# Pipeline d'entraînement 
## Fine-tuning MMS → baoulé sur Colab GPU T4

Dans **Exécution → Modifier le type d'exécution → Version de l'environnement d'exécution / Runtime Version**, sélectionnez **2026.07** (Python 3.12.13), puis un GPU. La [documentation officielle Colab](https://research.google.com/colaboratory/runtime-version-faq.html) liste cette version. Si ce choix n'apparaît pas, choisissez une version disponible en Python 3.12 ; le contrôle suivant affichera la version et s'arrêtera avant installation si elle est incompatible.

Les identifiants proviennent de votre notebook local, pas d'une vérification authentifiée du Hub. Les contrôles ci-dessous s'arrêtent si une ressource est inaccessible. Gardez suffisamment d'espace libre sur Drive (plusieurs Go).

```text
facebook/mms-tts-aka (générateur préentraîné)
  + discriminateur Meta full_models/aka/D_100000.pth
  → Tree-AI-lab/mms-tts-bau-baseline (conversion déjà réussie)
  → copie locale avec embeddings adaptés au tokenizer baoulé
  → entraînement sur Tree-AI-lab/bau-tts-monospeaker
  → Tree-AI-lab/mms-tts-bau-finetuned
```
---

## 🔗 Dépendance avec `pipeline-tts_baoulé.ipynb`

Ce notebook est le **second maillon** d'un pipeline en deux notebooks. Il ne fait qu'entraîner : il ne prépare ni ne publie ni le dataset ni le tokenizer, il les consomme en lecture seule.

| | `pipeline-tts_baoulé.ipynb` (préparation) | `training_mms_tts_baoule.ipynb` (ce fichier — entraînement) |
|---|---|---|
| **Rôle** | Explore `google/WaxalNLP` (`bau_tts`), nettoie le texte, filtre le locuteur JH, construit le vocabulaire baoulé, convertit le checkpoint donneur akan | Charge ces ressources déjà publiées, transfère les embeddings, lance et reprend l'entraînement, publie le modèle final |
| **Produit** | `Tree-AI-lab/bau-tts-monospeaker` (dataset), `Tree-AI-lab/mms-tts-bau-tokenizer` (tokenizer), `Tree-AI-lab/mms-tts-bau-baseline` (checkpoint donneur + discriminateur) | `Tree-AI-lab/mms-tts-bau-finetuned` (modèle final) |
| **Consomme** | `facebook/mms-tts-aka`, `google/WaxalNLP` | Les 3 dépôts publiés par le notebook de préparation |


In [ ]:
import sys, subprocess
print("Version Python :", sys.version)
assert (3, 10) <= sys.version_info[:2] <= (3, 12), "Choisir Runtime Version 2026.07 (Python 3.12) dans Exécution → Modifier le type d’exécution, puis relancer cette cellule."
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "torchaudio==2.5.1", "torchvision==0.20.1", "transformers==4.44.2", "datasets[audio]==3.6.0", "accelerate==0.34.2", "huggingface_hub==0.34.4", "numpy==1.26.4", "matplotlib==3.9.4", "librosa==0.10.2.post1", "tensorboard", "Cython", "soundfile"], check=True)


In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall",
    "--no-cache-dir",
    "numpy==1.26.4",
    "pandas==2.2.3",
    "pyarrow==17.0.0",
], check=True)

In [ ]:
import sys
import numpy
import pandas
import pyarrow

print("Python :", sys.version)
print("NumPy :", numpy.__version__)
print("pandas :", pandas.__version__)
print("PyArrow :", pyarrow.__version__)

assert numpy.__version__ == "1.26.4"
assert pandas.__version__ == "2.2.3"
assert pyarrow.__version__ == "17.0.0"

print("Environnement binaire cohérent.")

### Étape 2 — Redémarrer la session Python

**Après cette installation, redémarrez la session Python** (menu Exécution), puis continuez à la cellule suivante. Ne relancez pas l'installation après ce redémarrage.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import torch
assert torch.cuda.is_available(), "Activer un GPU Colab avant de continuer."
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
from huggingface_hub import notebook_login
notebook_login()  # Token autorisé à lire les sources et écrire le modèle dans Tree-AI-lab.
REPO = Path('/content/finetune-hf-vits')
COMMIT = '6f3f51f4d667f5c3eef89484d151ffd39d2c2b89'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ylacombe/finetune-hf-vits.git', str(REPO)], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True).strip() == '', 'Le clone contient des modifications : utiliser un clone propre.'
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
os.chdir(REPO)
(REPO / 'monotonic_align/monotonic_align').mkdir(exist_ok=True)
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=REPO / 'monotonic_align', check=True)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, 'run_vits_finetuning.py', '--help'], check=True, stdout=subprocess.DEVNULL)
DATASET = 'Tree-AI-lab/bau-tts-monospeaker'
TOKENIZER = 'Tree-AI-lab/mms-tts-bau-tokenizer'
BASELINE = 'Tree-AI-lab/mms-tts-bau-baseline'  # facebook/mms-tts-aka + discriminateur ; conversion déjà réussie
TARGET = 'Tree-AI-lab/mms-tts-bau-finetuned'
RUN = Path('/content/drive/MyDrive/tts-baoule/run-001')
RUN.mkdir(parents=True, exist_ok=True)
assert TARGET not in (TOKENIZER, BASELINE, 'facebook/mms-tts-aka')


### Étape 3 — Vérifier les ressources déjà publiées (dataset + tokenizer)

## Vérifier les ressources déjà publiées
Ce contrôle charge seulement les splits d'entraînement et de validation. Le test reste réservé à l'évaluation finale. Le rééchantillonnage à 16 kHz se fait localement, sans modification du Hub. Les fichiers longs seront exclus par le script, pas découpés arbitrairement.


In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.errors import RepositoryNotFoundError
from datasets import load_dataset, Audio as DatasetAudio
from transformers import AutoTokenizer
from collections import Counter
api = HfApi()
data_info = api.dataset_info(DATASET)
tok_info = api.model_info(TOKENIZER)
tok_path = snapshot_download(TOKENIZER, revision=tok_info.sha)
tokenizer = AutoTokenizer.from_pretrained(tok_path)
assert not tokenizer.is_uroman and not tokenizer.phonemize
assert tokenizer.pad_token_id is not None
splits = {}
for name in ('train', 'validation'):
    ds = load_dataset(DATASET, split=name, revision=data_info.sha)
    assert {'audio', 'text', 'speaker_id'} <= set(ds.column_names)
    assert set(ds['speaker_id']) == {'JH'}, f"Locuteurs inattendus : {Counter(ds['speaker_id'])}"
    assert len(ds) > 0
    splits[name] = ds.cast_column('audio', DatasetAudio(sampling_rate=16000))
    print(name, len(ds), Counter(ds['speaker_id']))
    # Contrôle des caractères avant la normalisation interne, qui peut les supprimer.
    missing = Counter(ch for t in ds['text'] for ch in t.lower() if ch not in tokenizer.get_vocab())
    assert not missing, f"Caractères non couverts dans {name} : {missing}"
    sample = splits[name][0]['audio']
    assert sample['sampling_rate'] == 16000 and len(sample['array']) > 0
print('Dataset et tokenizer accessibles ; exemple audio décodé à 16 kHz.')


### Étape 4 — Préparer le modèle initial en local (transfert d'embeddings caractère par caractère)

## Préparer le modèle initial localement
La baseline doit contenir le générateur **et le discriminateur**. Si elle est absente ou privée sans accès, le contrôle demandera de vérifier ce point avant une conversion locale. Aucun modèle de départ n'est envoyé au Hub.

Le tokenizer baoulé attribue de nouveaux identifiants aux caractères. On copie donc les embeddings akan par **caractère**, ainsi que les tokens spéciaux correspondants ; seuls les caractères nouveaux sont initialisés aléatoirement. Le tokenizer publié reste inchangé.


In [ ]:
from utils import VitsConfig, VitsModelForPreTraining, VitsFeatureExtractor
from transformers import set_seed
set_seed(456)
# Mettre True seulement si la baseline n'a effectivement jamais été publiée.
CONVERT_BASELINE_LOCALLY = False
if CONVERT_BASELINE_LOCALLY:
    from huggingface_hub import hf_hub_download
    import shutil
    generator = REPO / 'aka-generator-local'
    shutil.copytree(snapshot_download('facebook/mms-tts-aka'), generator, dirs_exist_ok=True)
    cfg = json.loads((generator / 'config.json').read_text())
    cfg['pad_token_id'] = AutoTokenizer.from_pretrained(generator).pad_token_id
    (generator / 'config.json').write_text(json.dumps(cfg))
    disc = hf_hub_download('facebook/mms-tts', subfolder='full_models/aka', filename='D_100000.pth')
    baseline_path = REPO / 'aka-training-local'
    subprocess.run([sys.executable, 'convert_original_discriminator_checkpoint.py', '--checkpoint_path', disc, '--generator_checkpoint_path', str(generator), '--pytorch_dump_folder_path', str(baseline_path)], check=True)
    baseline_revision = 'local conversion from facebook/mms-tts-aka'
else:
    try:
        info = api.model_info(BASELINE)
    except RepositoryNotFoundError as e:
        raise RuntimeError('Baseline introuvable ou privée : vérifier le nom et les droits. Si elle ne fut jamais publiée, activer CONVERT_BASELINE_LOCALLY dans cette cellule.') from e
    baseline_revision = info.sha
    baseline_path = snapshot_download(BASELINE, revision=info.sha)

old_tokenizer = AutoTokenizer.from_pretrained(baseline_path)
model, loading = VitsModelForPreTraining.from_pretrained(baseline_path, output_loading_info=True)
assert not loading['missing_keys'], f"Baseline incomplète : {loading['missing_keys']}"
assert model.config.num_speakers == 1 and model.config.sampling_rate == 16000
old_vocab = old_tokenizer.get_vocab()
new_vocab = tokenizer.get_vocab()
assert set(new_vocab.values()) == set(range(len(tokenizer))), 'Identifiants non contigus.'
old_weight = model.get_input_embeddings().weight.detach().clone()
old_size = old_weight.shape[0]
original_pad_id = model.config.pad_token_id
# get_vocab() inclut aussi les tokens ajoutés, parfois sans poids dans le modèle.
unbacked = {token: idx for token, idx in old_vocab.items() if not 0 <= idx < old_size}
print('Tokens source sans embedding :', unbacked)
model.resize_token_embeddings(len(tokenizer))
new_embedding = model.get_input_embeddings()
with torch.no_grad():
    new_embedding.weight.normal_(mean=0.0, std=model.config.initializer_range)
    copied = 0
    for token, new_id in new_vocab.items():
        old_id = old_vocab.get(token)
        if old_id is not None and 0 <= old_id < old_size:
            new_embedding.weight[new_id].copy_(old_weight[old_id])
            copied += 1

    # Le blank/padding doit provenir d'une ligne réellement présente.
    source_pad_id = old_tokenizer.pad_token_id
    if source_pad_id is None or not 0 <= source_pad_id < old_size:
        source_pad_id = original_pad_id
    if source_pad_id is None or not 0 <= source_pad_id < old_size:
        raise ValueError('Aucun embedding blank/pad source valide : vérifier config.json et vocab.json.')
    new_embedding.weight[tokenizer.pad_token_id].copy_(old_weight[source_pad_id])

    # Un <unk> ajouté au tokenizer source n'a pas forcément de poids préentraîné.
    source_unk_id = old_tokenizer.unk_token_id
    if (source_unk_id is not None and 0 <= source_unk_id < old_size
            and tokenizer.unk_token_id is not None):
        new_embedding.weight[tokenizer.unk_token_id].copy_(old_weight[source_unk_id])
    elif tokenizer.unk_token_id is not None:
        print('Token inconnu cible : initialisation aléatoire conservée (pas de poids source).')
new_embedding.padding_idx = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id
INITIAL = REPO / 'bau-initial-local'
model.save_pretrained(INITIAL)
tokenizer.save_pretrained(INITIAL)
VitsFeatureExtractor.from_pretrained(baseline_path).save_pretrained(INITIAL)
print(f'{copied} tokens communs transférés ; vocabulaire final : {len(tokenizer)}.')
del model, old_weight


### Étape 5 — Construire la configuration d'entraînement et patcher le script

## Configuration et sauvegardes
On conserve les 200 époques et le taux d'apprentissage de votre configuration. Le batch passe à 4 et la précision à FP32 pour le premier entraînement ; la taille du batch et la qualité seront à ajuster après écoute. 200 époques sont une hypothèse de départ, pas une garantie de qualité.

Les états d'entraînement sont sauvegardés tous les 100 pas dans Drive. Le script d'origine restaure les poids et optimiseurs, mais recommence au début de l'époque interrompue : la reprise n'est pas une reproduction exacte des lots. Ne changez pas les données, versions ou hyperparamètres au milieu d'une reprise.

Un marqueur local permet d'éviter de choisir une sauvegarde interrompue. Il n'assure pas que Google Drive a terminé la synchronisation distante : attendez la fin des écritures avant de fermer Colab.


In [ ]:
# Copie du script : marque les sauvegardes terminées, sans modifier le clone suivi par Git.
source = (REPO / 'run_vits_finetuning.py').read_text()
needle = '                        accelerator.save_state(save_path)'
assert source.count(needle) == 1
source = source.replace(needle, needle + '\n                        with open(os.path.join(save_path, "COMPLETE"), "w") as marker:\n                            marker.write(str(global_step))')
# Fige également la révision du dataset lue par le script.
needle = '            data_args.dataset_config_name,\n'
assert source.count(needle) == 2
source = source.replace(needle, needle + f'            revision={data_info.sha!r},\n')
# Limite la taille des lots Arrow : les sources contiennent aussi de longs audios.
source = source.replace('            num_proc=num_workers,', '            num_proc=num_workers,\n            writer_batch_size=8,')
SCRIPT = REPO / 'run_baoule_finetuning.py'
SCRIPT.write_text(source)
config = {
    'project_name': 'mms_baoule_finetuning', 'push_to_hub': False,
    'hub_model_id': TARGET, 'report_to': ['tensorboard'],
    'overwrite_output_dir': False, 'output_dir': str(RUN / 'training'),
    'logging_dir': str(RUN / 'logs'),
    'dataset_name': DATASET, 'audio_column_name': 'audio', 'text_column_name': 'text',
    'train_split_name': 'train', 'eval_split_name': 'validation',
    'speaker_id_column_name': None,
    'full_generation_sample_text': "Kɛ ɔ fɛ i aeroport Félix Houphouet-Boigny su lele mon fa ju",
    'max_duration_in_seconds': 30, 'min_duration_in_seconds': 1.0, 'max_tokens_length': 500,
    'model_name_or_path': str(INITIAL), 'tokenizer_name': str(INITIAL),
    'override_vocabulary_embeddings': False, 'preprocessing_num_workers': 1,
    'do_train': True, 'num_train_epochs': 200, 'gradient_accumulation_steps': 1,
    'gradient_checkpointing': False, 'per_device_train_batch_size': 4,
    'learning_rate': 2e-5, 'adam_beta1': 0.8, 'adam_beta2': 0.99, 'warmup_ratio': 0.01,
    'group_by_length': False, 'do_eval': True, 'eval_steps': 50,
    'per_device_eval_batch_size': 2, 'max_eval_samples': 25,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3, 'weight_fmaps': 1, 'weight_gen': 1, 'weight_kl': 1.5,
    'weight_duration': 1, 'weight_mel': 35,
    'fp16': False, 'seed': 456, 'save_steps': 100, 'save_total_limit': 3, 'logging_steps': 10,
}
manifest = {'dataset': DATASET, 'dataset_revision': data_info.sha, 'tokenizer': TOKENIZER,
            'tokenizer_revision': tok_info.sha, 'baseline_revision': baseline_revision,
            'code_commit': COMMIT, 'config': config}
manifest_file = RUN / 'run_manifest.json'
if manifest_file.exists():
    assert json.loads(manifest_file.read_text()) == manifest, 'Configuration ou sources modifiées : choisir un nouveau RUN pour un nouvel entraînement.'
else:
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(RUN / 'environment.txt').open('w'), check=True)


In [ ]:
# Réappliquer le correctif après la recréation du script
script_source = SCRIPT.read_text()

old_call = 'speaker_id=batch["speaker_id"],'
new_call = 'speaker_id=batch.get("speaker_id"),'

old_count = script_source.count(old_call)
new_count = script_source.count(new_call)

print("Appels non corrigés :", old_count)
print("Appels déjà corrigés :", new_count)

assert (old_count, new_count) in ((3, 0), (0, 3)), (
    old_count,
    new_count,
)

if old_count == 3:
    script_source = script_source.replace(old_call, new_call)
    SCRIPT.write_text(script_source)

compile(SCRIPT.read_text(), str(SCRIPT), "exec")

assert SCRIPT.read_text().count(old_call) == 0
assert SCRIPT.read_text().count(new_call) == 3

print("Correctif mono-locuteur actif.")

In [ ]:
# Sauvegardes Drive durables : moins fréquentes et synchronisées avant de continuer.
script_source = SCRIPT.read_text()
save_log = '                        logger.info(f"Saved state to {save_path}")'
sync_marker = "Synchronizing checkpoint with mounted storage"
early_complete = (
    '                        with open(os.path.join(save_path, "COMPLETE"), "w") as marker:\n'
    '                            marker.write(str(global_step))\n'
)
if sync_marker not in script_source:
    assert script_source.count(save_log) == 1
    assert script_source.count(early_complete) == 1
    script_source = script_source.replace(early_complete, '')
    sync_block = (
        '                        logger.info("Synchronizing checkpoint with mounted storage")\n'
        '                        os.sync()\n'
        '                        import time\n'
        '                        time.sleep(60)\n'
        '                        with open(os.path.join(save_path, "COMPLETE"), "w") as marker:\n'
        '                            marker.write(str(global_step))\n'
        '                            marker.flush()\n'
        '                            os.fsync(marker.fileno())\n'
        '                        os.sync()\n'
    )
    script_source = script_source.replace(save_log, sync_block + save_log)
    SCRIPT.write_text(script_source)
compile(SCRIPT.read_text(), str(SCRIPT), 'exec')
assert SCRIPT.read_text().count(sync_marker) == 1
assert SCRIPT.read_text().index(sync_marker) < SCRIPT.read_text().index('open(os.path.join(save_path, "COMPLETE"')
# Surcharge de reprise : le manifeste historique reste inchangé.
config['save_steps'] = 500
config['save_total_limit'] = 4
print('Sauvegardes configurées tous les 500 pas, avec synchronisation et 4 copies maximum.')


### Étape 6 — Test de fumée (2 pas) avant l'entraînement complet

## Essai de deux pas
L'essai utilise un dossier séparé et ne publie rien. Vérifiez qu'il se termine sans erreur et que les pertes sont finies avant la cellule d'entraînement. Il valide l'exécution, pas la qualité de la voix. Si CUDA manque de mémoire, réduire le batch avant de créer un nouveau RUN.


In [ ]:
import uuid
smoke = dict(config)
smoke.update(output_dir=f'/content/bau-smoke-{uuid.uuid4().hex[:8]}', logging_dir='/content/bau-smoke-logs',
             max_steps=2, max_train_samples=8, max_eval_samples=2,
             eval_steps=1, save_steps=1, logging_steps=1, report_to=[])
smoke_path = REPO / 'smoke_baoule.json'
smoke_path.write_text(json.dumps(smoke, ensure_ascii=False, indent=2))
# Affiche stdout et stderr dans Colab, en conservant une copie du diagnostic.
smoke_log = RUN / 'smoke_latest.log'
with smoke_log.open('w', encoding='utf-8') as log:
    with subprocess.Popen(
        [sys.executable, '-u', str(SCRIPT), str(smoke_path)],
        cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
print(f'Journal de cet essai : {smoke_log}')
if returncode != 0:
    print('\n--- Dernières lignes du diagnostic ---')
    print('\n'.join(smoke_log.read_text().splitlines()[-80:]))
    raise RuntimeError(f"L'essai a échoué (code {returncode}). Voir le diagnostic ci-dessus et {smoke_log}.")


### Étape 7 — Nettoyer l'essai et vérifier l'espace disque

## Nettoyer l'essai et contrôler l'espace local
Cette cellule supprime uniquement les sorties temporaires `bau-smoke-*` et le cache d'installation pip. Elle conserve le cache Hugging Face, le modèle initial et tous les checkpoints stockés dans Drive. Exécutez-la après l'essai, puis avant une reprise si le disque local est presque plein.


In [ ]:
import shutil
smoke_dirs = list(Path('/content').glob('bau-smoke-*'))
for smoke_dir in smoke_dirs:
    if smoke_dir.is_dir() and smoke_dir.parent == Path('/content'):
        shutil.rmtree(smoke_dir)
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)
total, used, free = shutil.disk_usage('/content')
print('Essais supprimés :', [p.name for p in smoke_dirs])
print(f'Espace local : {used / 2**30:.1f} Gio utilisés, {free / 2**30:.1f} Gio libres')
assert free >= 15 * 2**30, 'Moins de 15 Gio libres : redémarrer avec une nouvelle session Colab puis reprendre depuis Drive.'


### Étape 8 — Détecter les checkpoints et lancer / reprendre l'entraînement

## Lancer ou reprendre le fine-tuning
Après une déconnexion, réexécutez les cellules de préparation avec le même RUN. Cette cellule choisit explicitement le dernier checkpoint marqué complet. Si la session a disparu avant le premier checkpoint, aucun état n'est récupérable ; choisissez un nouveau RUN pour repartir du modèle initial.

L'entraînement tourne dans Colab, pas dans VS Code. La durée dépend du GPU et du nombre de clips retenus après les filtres durée/tokens. Les métriques et écoutes TensorBoard servent à détecter une dégradation ou un surapprentissage.


In [ ]:
from pathlib import Path
import shutil

output = RUN / "training"
print("Dossier Drive :", output)
print("Existe :", output.exists())

all_checkpoints = sorted(
    (p for p in output.glob("checkpoint-*") if p.name.split("-")[-1].isdigit()),
    key=lambda p: int(p.name.split("-")[-1]),
) if output.exists() else []

complete_checkpoints = [p for p in all_checkpoints if (p / "COMPLETE").exists()]
incomplete_checkpoints = [p for p in all_checkpoints if not (p / "COMPLETE").exists()]

print("Checkpoints complets :")
for checkpoint in complete_checkpoints:
    print(" \u2713", checkpoint.name)

print("Checkpoints interrompus :")
for checkpoint in incomplete_checkpoints:
    print(" \u2717", checkpoint.name)

# Supprime les checkpoints interrompus : le script voudrait sinon les recréer
# en repartant du dernier checkpoint complet, ce qui provoquerait un conflit.
for checkpoint in incomplete_checkpoints:
    print("Suppression du checkpoint incomplet :", checkpoint)
    shutil.rmtree(checkpoint)

if complete_checkpoints:
    print("Reprise possible depuis :", complete_checkpoints[-1])
elif all_checkpoints:
    print("Aucun checkpoint complet restant après nettoyage : l'entraînement repartira du modèle initial.")
else:
    print("Aucun checkpoint existant : premier lancement, l'entraînement démarrera depuis le modèle initial.")

In [ ]:
output = Path(config['output_dir'])
checkpoints = sorted((p for p in output.glob('checkpoint-*') if (p / 'COMPLETE').exists() and p.name.split('-')[-1].isdigit()), key=lambda p: int(p.name.split('-')[-1]))
if checkpoints:
    config['resume_from_checkpoint'] = str(checkpoints[-1])
    print('Reprise :', checkpoints[-1], flush=True)
elif output.exists() and any(output.iterdir()):
    raise RuntimeError('Dossier non vide sans checkpoint complet. Choisir un nouveau RUN ; conserver celui-ci pour diagnostic.')
config_path = RUN / 'finetune_baoule.json'
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2))
# Affiche les logs immédiatement et conserve une copie dans Google Drive.
training_log = RUN / 'training_latest.log'
print('Lancement du script :', SCRIPT, flush=True)
print('Journal :', training_log, flush=True)
with training_log.open('a', encoding='utf-8') as log:
    log.write('\n\n=== Nouveau lancement / reprise ===\n')
    log.flush()
    with subprocess.Popen(
        [sys.executable, '-u', str(SCRIPT), str(config_path)],
        cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
if returncode != 0:
    print('\n--- Dernières lignes du journal ---')
    print('\n'.join(training_log.read_text().splitlines()[-100:]))
    raise RuntimeError(f'Entraînement interrompu avec le code {returncode}. Voir {training_log}.')
print('Entraînement terminé correctement.', flush=True)


### Étape 9 — Écouter un échantillon puis publier le modèle final

## Écouter puis publier le modèle final
Cette cellule d'écoute utilise un texte de validation déjà normalisé. Pour vos propres phrases, appliquez le même nettoyage que dans votre notebook original (NFC, apostrophes et espaces notamment). Évaluez ensuite sur des phrases jamais vues, et avec un locuteur baoulé.

La dernière cellule publie uniquement dans `Tree-AI-lab/mms-tts-bau-finetuned`. L'exécuter actualise ce dépôt s'il existe déjà. Les checkpoints avec optimiseurs restent dans Drive ; le dépôt final sert à la synthèse.


In [ ]:
from transformers import VitsModel
from IPython.display import Audio, display
final_model = VitsModel.from_pretrained(config['output_dir']).to('cuda').eval()
final_tokenizer = AutoTokenizer.from_pretrained(config['output_dir'])
text = splits['validation'][0]['text']
print(text)
set_seed(456)
inputs = final_tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    waveform = final_model(**inputs).waveform[0].cpu().numpy()
display(Audio(waveform, rate=final_model.config.sampling_rate))


In [ ]:
# Export dédié : aucun checkpoint d'optimiseur ni jeton d'accès dans l'envoi.
EXPORT = REPO / 'bau-inference-export'
final_model.cpu().save_pretrained(EXPORT)
final_tokenizer.save_pretrained(EXPORT)
VitsFeatureExtractor.from_pretrained(config['output_dir']).save_pretrained(EXPORT)
card = """---
language:
- bci
license: cc-by-nc-4.0
base_model: facebook/mms-tts-aka
datasets:
- Tree-AI-lab/bau-tts-monospeaker
pipeline_tag: text-to-speech
---
# MMS TTS baoulé
Adaptation du modèle akan facebook/mms-tts-aka sur la voix JH du sous-ensemble bau_tts de Google WaxalNLP, préparé dans Tree-AI-lab/bau-tts-monospeaker.
Tokenizer : Tree-AI-lab/mms-tts-bau-tokenizer. Audio : 16 kHz. Embeddings communs transférés par caractère.
La qualité n'a pas fait l'objet d'une évaluation formelle. Les clips sont filtrés à 1–30 secondes et 500 tokens maximum.
Le modèle de base est sous licence CC BY-NC 4.0.
"""
(EXPORT / 'README.md').write_text(card)
assert TARGET == 'Tree-AI-lab/mms-tts-bau-finetuned'
api.create_repo(TARGET, repo_type='model', exist_ok=True)
api.upload_folder(repo_id=TARGET, repo_type='model', folder_path=EXPORT,
                  commit_message='Publish Baoule MMS TTS fine-tuned model and tokenizer')
# Vérifie que le dépôt publié est rechargeable.
published_tokenizer = AutoTokenizer.from_pretrained(TARGET)
published_model = VitsModel.from_pretrained(TARGET)
assert published_tokenizer.get_vocab() == final_tokenizer.get_vocab()
assert published_model.config.vocab_size == len(published_tokenizer)
print('Modèle publié : https://huggingface.co/' + TARGET)


### Sources et provenance

Sources : [dépôt ylacombe](https://github.com/ylacombe/finetune-hf-vits), [modèle akan](https://huggingface.co/facebook/mms-tts-aka), [décodage audio Datasets 3.6](https://huggingface.co/docs/datasets/v3.6.0/en/audio_process). Le notebook historique local et les six pièces jointes de la conversation Claude fournissent les identifiants Tree-AI-lab et les choix de données. Aucune lecture authentifiée de ces dépôts ni exécution GPU n'a été réalisée lors de la création de ce fichier.
